# FAQ bot — two-stage information retrieval

1. **Stage 1 (dense retrieval):** a multilingual **bi-encoder** embeds the user question and document chunks; Chroma returns the top-`initial_k` candidates by cosine similarity (high **recall**).
2. **Stage 2 (reranking):** a **cross-encoder** scores each (query, chunk) pair jointly and reorders the list; you keep the top-`final_k` chunks (better **precision** for the generator).

This notebook mirrors the pattern in `notebooks/pablo/two_stage_ir_pipeline.ipynb`, wired to the project PDF under `data/`, with models that load without gated access.

In [ ]:
# Uncomment if needed:
# %pip install -q langchain-community langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers pypdf torch

## Paths and configuration

In [ ]:
from __future__ import annotations

import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Sequence

import torch
from sentence_transformers import CrossEncoder

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data").is_dir() and (p / "notebooks").is_dir():
            return p
    return here


PROJECT_ROOT = find_project_root()


@dataclass
class FaqIrConfig:
    pdf_path: Optional[Path] = None  # set when building config
    persist_directory: str = "./chroma_langchain_db_arthur_faq_2stage"
    collection_name: str = "faq_sindilojas_2stage"
    embedding_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    reranker_model_name: str = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
    chunk_size: int = 500
    chunk_overlap: int = 100
    initial_k: int = 20
    final_k: int = 5
    device: Optional[str] = None
    normalize_embeddings: bool = True
    reranker_batch_size: int = 8


def get_device(explicit: Optional[str] = None) -> str:
    if explicit:
        return explicit
    return "cuda" if torch.cuda.is_available() else "cpu"


config = FaqIrConfig(
    pdf_path=PROJECT_ROOT / "data" / "marcelo" / "sindilojas_2025_2026.pdf",
)
device = get_device(config.device)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PDF:", config.pdf_path)
print("PDF exists:", config.pdf_path.is_file())
print("Device:", device)
print("Embedding model:", config.embedding_model_name)
print("Reranker model:", config.reranker_model_name)

## Ingest PDF → chunks → Chroma

In [ ]:
loader = PyPDFLoader(str(config.pdf_path))
docs = loader.load()
print("Pages loaded:", len(docs))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)
print("Chunks:", len(all_splits))

In [ ]:
def deterministic_chunk_id(source: str, page: int, chunk_text: str) -> str:
    payload = f"{source}|{page}|{chunk_text.strip().lower()}"
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


embeddings = HuggingFaceEmbeddings(
    model_name=config.embedding_model_name,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": config.normalize_embeddings},
)

vector_store = Chroma(
    collection_name=config.collection_name,
    embedding_function=embeddings,
    persist_directory=config.persist_directory,
)

chunk_ids = [
    deterministic_chunk_id(
        str(chunk.metadata.get("source", "")),
        int(chunk.metadata.get("page", 0) or 0),
        chunk.page_content,
    )
    for chunk in all_splits
]

ids = vector_store.add_documents(documents=all_splits, ids=chunk_ids)
print("Indexed chunks:", len(ids))

## Two-stage retriever (dense + cross-encoder)

- `invoke(query)` runs stage 1 then stage 2.
- `as_runnable()` exposes a LangChain `Runnable` for LCEL chains.

In [ ]:
class TwoStageRetriever:
    """
    Stage 1: dense retrieval in Chroma (bi-encoder).
    Stage 2: cross-encoder reranking (sentence-transformers).
    """

    def __init__(
        self,
        vectorstore: Chroma,
        reranker_model_name: str,
        initial_k: int = 20,
        final_k: int = 5,
        device: Optional[str] = None,
        reranker_batch_size: int = 8,
    ) -> None:
        self.vectorstore = vectorstore
        self.initial_k = initial_k
        self.final_k = final_k
        self.device = get_device(device)
        self.reranker_batch_size = reranker_batch_size
        self.cross_encoder = CrossEncoder(
            reranker_model_name,
            device=self.device,
        )

    def _dense_retrieve(self, query: str) -> List[Document]:
        return self.vectorstore.similarity_search(query, k=self.initial_k)

    def _rerank(self, query: str, docs: Sequence[Document]) -> List[Document]:
        if not docs:
            return []
        pairs = [(query, d.page_content) for d in docs]
        scores = self.cross_encoder.predict(
            pairs,
            batch_size=self.reranker_batch_size,
            show_progress_bar=False,
        )
        scored: List[Document] = []
        for doc, score in zip(docs, scores):
            meta = dict(doc.metadata) if doc.metadata else {}
            meta["reranker_score"] = float(score)
            scored.append(Document(page_content=doc.page_content, metadata=meta))
        scored.sort(key=lambda d: d.metadata["reranker_score"], reverse=True)
        return scored[: self.final_k]

    def invoke(self, query: str) -> List[Document]:
        return self._rerank(query, self._dense_retrieve(query))

    def batch(self, queries: Sequence[str]) -> List[List[Document]]:
        return [self.invoke(q) for q in queries]

    def as_runnable(self) -> RunnableLambda:
        return RunnableLambda(self.invoke)


retriever = TwoStageRetriever(
    vectorstore=vector_store,
    reranker_model_name=config.reranker_model_name,
    initial_k=config.initial_k,
    final_k=config.final_k,
    device=config.device,
    reranker_batch_size=config.reranker_batch_size,
)
print("Two-stage retriever ready.")

## Try it (FAQ-style question)

In [ ]:
def print_stage1_vs_stage2(query: str, max_chars: int = 600) -> None:
    stage1 = retriever._dense_retrieve(query)
    stage2 = retriever.invoke(query)
    print("=== Stage 1 (dense, first 3) ===")
    for i, d in enumerate(stage1[:3], 1):
        print(f"\n[{i}] page={d.metadata.get('page', '?')}\n{d.page_content[:max_chars]}...")
    print("\n=== Stage 2 (reranked top final_k) ===")
    for i, d in enumerate(stage2, 1):
        s = d.metadata.get("reranker_score", float("nan"))
        print(f"\n[{i}] reranker_score={s:.4f} page={d.metadata.get('page', '?')}\n{d.page_content[:max_chars]}...")


q = (
    "Quais obrigações a empresa deve cumprir, em caso de parcelamento da rescisão?"
)
print_stage1_vs_stage2(q)